# Signal research playground

Interactive companion to the `intraday` app. Use this to eyeball signals on real candles,
run quick backtests, and iterate on new signal ideas before promoting them.
**Kernel:** pick **Python (TradingData)** - already registered (the venv at `C:\TradingData\venv`).

(in VS Code: *Select Kernel → Python Environments → TradingData venv*).

**Pipeline recap:** `trader download` fills the candle cache → signals are plugins in
`src/intraday/signals/` → `trader backtest` scores them net of Indian costs → survivors go
to the paper bot.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import intraday.signals as sig
from intraday.config import settings
from intraday.data import store
from intraday.signals import indicators
from intraday.signals.registry import Context, all_signals

INTERVAL = settings()["data"]["interval"]

# chart chrome: recessive grid/axes, ink for data, color only where it means something
INK, MUTED, GRID, BASELINE = "#0b0b0b", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, GOOD, CRITICAL = "#2a78d6", "#008300", "#d03b3b"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": BASELINE, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED,
    "ytick.color": MUTED, "font.family": "sans-serif", "figure.figsize": (11, 4.5),
})

sig.load_all()
pd.DataFrame(
    [(s.name, s.description, s.params) for s in all_signals().values()],
    columns=["signal", "description", "default params"],
).set_index("signal")

## Load candles for a symbol

In [ ]:
SYMBOL = "RELIANCE"

available = store.available_symbols(INTERVAL)
if SYMBOL not in available:
    SYMBOL = available[0]
    print(f"symbol not cached, using {SYMBOL}")

df = store.load(SYMBOL, INTERVAL)
print(f"{SYMBOL}: {len(df):,} candles, {df.index[0]} .. {df.index[-1]}")
df.tail(3)

## See a signal fire on the chart

Replays the last few sessions bar-by-bar through one signal and marks each firing
on the price line (green up-triangle = BUY, red down-triangle = SELL).

In [ ]:
SIGNAL, DAYS, WINDOW = "orb_breakout", 5, 300

def replay(df, signal_name, days, window=WINDOW):
    """Run one signal over the last `days` sessions; returns fired events."""
    spec = all_signals()[signal_name]
    day_list = sorted({d for d in df.index.date})[-days:]
    fires = []
    for pos, ts in enumerate(df.index):
        if ts.date() not in day_list:
            continue
        i = df.index.get_loc(ts)
        ev = spec.func(df.iloc[max(0, i + 1 - window): i + 1],
                       Context(symbol=SYMBOL, interval=INTERVAL, params=spec.params))
        if ev is not None:
            fires.append({"time": ts, "side": ev.side, "entry": ev.entry,
                          "stop": ev.stop, "target": ev.target, "reason": ev.reason})
    return pd.DataFrame(fires)

fires = replay(df, SIGNAL, DAYS)

view = df[df.index.date >= sorted({d for d in df.index.date})[-DAYS]]
x = range(len(view))  # positional x-axis: no overnight-gap flat lines
fig, ax = plt.subplots()
ax.plot(x, view["close"], color=INK, lw=1.2, label="close")
ax.plot(x, indicators.session_vwap(view), color=BLUE, lw=1.2, alpha=0.8, label="session VWAP")
pos_of = {ts: i for i, ts in enumerate(view.index)}
for _, f in fires.iterrows():
    buy = f.side == "BUY"
    ax.scatter(pos_of[f.time], f.entry, marker="^" if buy else "v", s=90, zorder=3,
               color=GOOD if buy else CRITICAL, edgecolor="#fcfcfb", linewidth=1.5)
day_starts = [i for i, ts in enumerate(view.index) if ts.time().hour == 9 and ts.time().minute == 15]
ax.set_xticks(day_starts)
ax.set_xticklabels([view.index[i].strftime("%d %b") for i in day_starts])
ax.set_title(f"{SYMBOL} — {SIGNAL}: {len(fires)} firings in last {DAYS} sessions",
             loc="left", fontsize=11)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()
fires

## Quick backtest

Same engine as `trader backtest`, callable inline. Costs (brokerage, STT, slippage, ...)
are already deducted — `net_pnl` is what would actually hit the account per Rs 50k trade.

In [ ]:
from intraday.backtest.engine import run_backtest
from intraday.backtest.stats import signal_stats

trades = run_backtest(
    symbols=available[:10],          # or a hand-picked list
    interval=INTERVAL,
    signal_names=["orb_breakout", "vwap_reversion", "cpr_breakout"],
    days_limit=30,
    workers=1,                        # keep 1 inside notebooks (Windows multiprocessing)
)
signal_stats(trades)

## Equity curve of one signal

In [ ]:
PICK = "orb_breakout"
g = trades[trades.signal == PICK].sort_values("exit_time")
if g.empty:
    print("no trades")
else:
    equity = g["net_pnl"].cumsum()
    fig, ax = plt.subplots()
    ax.plot(range(len(equity)), equity, color=BLUE, lw=1.6)
    ax.axhline(0, color=BASELINE, lw=1)
    ax.set_title(f"{PICK} — cumulative net P&L over {len(g)} trades (Rs)",
                 loc="left", fontsize=11)
    ax.set_xlabel("trade #")
    plt.tight_layout()
    plt.show()

## Prototype a new signal right here

Write it as a plain function, test it with `replay()` / `run_backtest` — then, when it
earns its keep, move it to a file in `src/intraday/signals/` with the `@signal(...)`
decorator and it becomes part of the app (CLI, scanner, paper bot) automatically.

In [ ]:
from intraday.signals import helpers
from intraday.signals.registry import SignalEvent, signal


@signal("my_experiment")  # comment the decorator out while iterating in-notebook
def my_experiment(df, ctx):
    """Example: buy a green candle closing above the last 30-bar high."""
    if len(df) < 40 or not helpers.entries_allowed(df):
        return None
    cur = df.iloc[-1]
    if cur.close > df["high"].iloc[-31:-1].max() and cur.close > cur.open:
        a = indicators.atr(df, 14).iloc[-1]
        return SignalEvent(side="BUY", entry=float(cur.close),
                           stop=float(cur.close - 1.5 * a),
                           target=float(cur.close + 3 * a), reason="30-bar high break")
    return None


replay(df, "my_experiment", days=5).tail()